In [1]:
from langchain_ollama import ChatOllama
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import SQLDatabaseToolkit
from langgraph.checkpoint.memory import InMemorySaver

llm = ChatOllama(model="llama3.1:latest", temperature=0)
db = SQLDatabase.from_uri("sqlite:///demo.db")
memory = InMemorySaver()

/Users/ariksarkar/Desktop/practice/gen-ai/.venv/lib/python3.12/site-packages/langgraph/checkpoint/serde/encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [2]:
db.run("""
    CREATE TABLE IF NOT EXISTS tasks(
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        title VARCHAR(100) NOT NULL,
        description TEXT NOT NULL,
        status VARCHAR(30) DEFAULT 'pending',
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    );
 """)

''

In [3]:
toolkit = SQLDatabaseToolkit(db=db, llm=llm)
tools = toolkit.get_tools()

In [5]:
from langchain_core.messages import SystemMessage, HumanMessage

system_prompt = SystemMessage(content="""
You are a SQL task assistant for a `tasks` table.

Rules:
- Use `ORDER BY created_at DESC LIMIT 10` for all task lists.
- After INSERT, UPDATE, or DELETE, run a SELECT to verify changes.
- If no records exist, reply: `No tasks found.`
- Return task lists as markdown tables.
- Keep responses short and precise.
- Prefer `id` over `title` when both exist.
- Do not generate unsupported SQL.

Schema:
tasks(
  id,
  title,
  description,
  status,
  created_at
)

Valid status values:
- pending
- in_progress
- completed

SQL Templates:

Create:
INSERT INTO tasks(title, description, status)
VALUES (?, ?, ?);

Read:
SELECT *
FROM tasks
WHERE ...
ORDER BY created_at DESC
LIMIT 10;

Update:
UPDATE tasks
SET status = ?
WHERE id = ? OR title = ?;

Delete:
DELETE FROM tasks
WHERE id = ? OR title = ?;
                              
Response Rules:
- Only display rows returned from the database.
- If rows exist, return a markdown table.
- If zero rows exist, not required to show the table format in this case and do not create mock data to display table records simply just return: "No tasks found"

Table format: (if records actually exists in the database)

| id | title | description | status | created_at |
|----|-------|-------------|--------|------------|
""")

In [6]:
system_prompt

SystemMessage(content='\nYou are a SQL task assistant for a `tasks` table.\n\nRules:\n- Use `ORDER BY created_at DESC LIMIT 10` for all task lists.\n- After INSERT, UPDATE, or DELETE, run a SELECT to verify changes.\n- If no records exist, reply: `No tasks found.`\n- Return task lists as markdown tables.\n- Keep responses short and precise.\n- Prefer `id` over `title` when both exist.\n- Do not generate unsupported SQL.\n\nSchema:\ntasks(\n  id,\n  title,\n  description,\n  status,\n  created_at\n)\n\nValid status values:\n- pending\n- in_progress\n- completed\n\nSQL Templates:\n\nCreate:\nINSERT INTO tasks(title, description, status)\nVALUES (?, ?, ?);\n\nRead:\nSELECT *\nFROM tasks\nWHERE ...\nORDER BY created_at DESC\nLIMIT 10;\n\nUpdate:\nUPDATE tasks\nSET status = ?\nWHERE id = ? OR title = ?;\n\nDelete:\nDELETE FROM tasks\nWHERE id = ? OR title = ?;\n\nResponse Rules:\n- Only display rows returned from the database.\n- If rows exist, return a markdown table.\n- If zero rows exist

In [7]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware

llm_summarizer = ChatOllama(model="qwen2.5-coder:7b", temperature=0) # Use cheaper model for summaries
summarizer = SummarizationMiddleware(
    model=llm_summarizer, 
    trigger=("messages", 10), # Trigger summarization when 50 messages is reached
    summary_prompt="""
        Summarize for SQL generation.

        Keep:
        - intent
        - tables
        - filters
        - metrics
        - dates
        - grouping
        - ordering
        - limits
        - sql errors
        - dialect

        Short structured summary only.

        Format:
        INTENT:
        TABLES:
        FILTERS:
        METRICS:
        TIME:
        ERRORS:
        """
)

agent = create_agent(
    model=llm,
    tools=[]+tools,
    system_prompt=system_prompt,
    checkpointer=memory,
    debug=True,
    middleware=[summarizer]
)

config = {"configurable": {"thread_id": "arik555"}}

In [8]:
message = {"messages": [
    HumanMessage(content="give me all the tasks")
]}
resp = agent.invoke(input=message, config=config)

[values] {'messages': [HumanMessage(content='give me all the tasks', additional_kwargs={}, response_metadata={}, id='e20003ad-d29a-43c6-8ab9-ce9602df4a54')]}
[updates] {'SummarizationMiddleware.before_model': None}
[updates] {'model': {'messages': [AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.1:latest', 'created_at': '2026-05-13T08:41:06.861229Z', 'done': True, 'done_reason': 'stop', 'total_duration': 5809371125, 'load_duration': 807923459, 'prompt_eval_count': 763, 'prompt_eval_duration': 3622095916, 'eval_count': 29, 'eval_duration': 1370568666, 'logprobs': None, 'model_name': 'llama3.1:latest', 'model_provider': 'ollama'}, id='lc_run--019e207f-197a-7900-ba8d-27ab01b8058f-0', tool_calls=[{'name': 'sql_db_query', 'args': {'query': 'SELECT * FROM tasks ORDER BY created_at DESC LIMIT 10'}, 'id': '2526d664-0788-43f4-997f-77fb24582f02', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 763, 'output_tokens': 29, 'total_tokens':

In [9]:
print(resp["messages"][-1].content)

| id | title | description | status | created_at |
|----|-------|-------------|--------|------------|
| 1  | Study RAG pipeline architecture | Learn about the RAG pipeline architecture | pending | 2026-05-13 08:11:43 |


In [10]:
message = {"messages": [
    HumanMessage(content="Ok, create one task to look for best authentic bengali restaurants in Kolkata")
]}
resp = agent.invoke(input=message, config=config)

[values] {'messages': [HumanMessage(content='give me all the tasks', additional_kwargs={}, response_metadata={}, id='e20003ad-d29a-43c6-8ab9-ce9602df4a54'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.1:latest', 'created_at': '2026-05-13T08:41:06.861229Z', 'done': True, 'done_reason': 'stop', 'total_duration': 5809371125, 'load_duration': 807923459, 'prompt_eval_count': 763, 'prompt_eval_duration': 3622095916, 'eval_count': 29, 'eval_duration': 1370568666, 'logprobs': None, 'model_name': 'llama3.1:latest', 'model_provider': 'ollama'}, id='lc_run--019e207f-197a-7900-ba8d-27ab01b8058f-0', tool_calls=[{'name': 'sql_db_query', 'args': {'query': 'SELECT * FROM tasks ORDER BY created_at DESC LIMIT 10'}, 'id': '2526d664-0788-43f4-997f-77fb24582f02', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 763, 'output_tokens': 29, 'total_tokens': 792}), ToolMessage(content="[(1, 'Study RAG pipeline architecture', 'Learn about the RAG pi

In [11]:
print(resp["messages"][-1].content)

| id | title | description | status | created_at |
|----|-------|-------------|--------|------------|
| 2  | Best Authentic Bengali Restaurants in Kolkata | Looking for the best authentic Bengali restaurants in Kolkata | pending | 2026-05-13 08:41:14 |
| 1  | Study RAG pipeline architecture | Learn about the RAG pipeline architecture | pending | 2026-05-13 08:11:43 |


In [14]:
message = {"messages": [
    HumanMessage(content="I will check the resturants in Kolkata right now, so make it in progress.")
]}
resp = agent.invoke(input=message, config=config)

[values] {'messages': [HumanMessage(content='give me all the tasks', additional_kwargs={}, response_metadata={}, id='e20003ad-d29a-43c6-8ab9-ce9602df4a54'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.1:latest', 'created_at': '2026-05-13T08:41:06.861229Z', 'done': True, 'done_reason': 'stop', 'total_duration': 5809371125, 'load_duration': 807923459, 'prompt_eval_count': 763, 'prompt_eval_duration': 3622095916, 'eval_count': 29, 'eval_duration': 1370568666, 'logprobs': None, 'model_name': 'llama3.1:latest', 'model_provider': 'ollama'}, id='lc_run--019e207f-197a-7900-ba8d-27ab01b8058f-0', tool_calls=[{'name': 'sql_db_query', 'args': {'query': 'SELECT * FROM tasks ORDER BY created_at DESC LIMIT 10'}, 'id': '2526d664-0788-43f4-997f-77fb24582f02', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 763, 'output_tokens': 29, 'total_tokens': 792}), ToolMessage(content="[(1, 'Study RAG pipeline architecture', 'Learn about the RAG pi

In [15]:
print(resp["messages"][-1].content)

| id | title | description | status | created_at |
|----|-------|-------------|--------|------------|
| 2  | Best Authentic Bengali Restaurants in Kolkata | Looking for the best authentic Bengali restaurants in Kolkata | in_progress | 2026-05-13 08:41:14 |


In [16]:
message = {"messages": [
    HumanMessage(content="I have studied the basic of RAG pipleline.")
]}
resp = agent.invoke(input=message, config=config)

[values] {'messages': [HumanMessage(content='give me all the tasks', additional_kwargs={}, response_metadata={}, id='e20003ad-d29a-43c6-8ab9-ce9602df4a54'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.1:latest', 'created_at': '2026-05-13T08:41:06.861229Z', 'done': True, 'done_reason': 'stop', 'total_duration': 5809371125, 'load_duration': 807923459, 'prompt_eval_count': 763, 'prompt_eval_duration': 3622095916, 'eval_count': 29, 'eval_duration': 1370568666, 'logprobs': None, 'model_name': 'llama3.1:latest', 'model_provider': 'ollama'}, id='lc_run--019e207f-197a-7900-ba8d-27ab01b8058f-0', tool_calls=[{'name': 'sql_db_query', 'args': {'query': 'SELECT * FROM tasks ORDER BY created_at DESC LIMIT 10'}, 'id': '2526d664-0788-43f4-997f-77fb24582f02', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 763, 'output_tokens': 29, 'total_tokens': 792}), ToolMessage(content="[(1, 'Study RAG pipeline architecture', 'Learn about the RAG pi

In [17]:
print(resp["messages"][-1].content)

| id | title | description | status | created_at |
|----|-------|-------------|--------|------------|
| 2  | Best Authentic Bengali Restaurants in Kolkata | Looking for the best authentic Bengali restaurants in Kolkata | in_progress | 2026-05-13 08:41:14 |
| 1  | Study RAG pipeline architecture | Learn about the RAG pipeline architecture | completed | 2026-05-13 08:11:43 |
